# Can AI improve exoplanet candidate screening?

ASTR 12600 | Research notebook. AI assists with writing code; established BLS is not a new ML model. The random forest predicts NASA's *historical Robovetter labels*, not confirmed planets.

Run **Runtime > Run all** in Google Colab. NASA access is required for the last experiment; synthetic experiments run without the NASA catalog.


In [ ]:
%pip -q install numpy pandas scikit-learn astropy matplotlib requests
from pathlib import Path
Path('results').mkdir(exist_ok=True)


## 1. Reproducing a published transit-search method

These examples use simulated flux. A strong periodicity peak alone is not evidence of an actual planet: inspect the noise-only control.


In [ ]:
%%writefile exoplanet_detector.py
"""Reproducible transit-search demonstration (not a discovery claim).

AI authored; BLS is the published method of Kovacs, Zucker and Mazeh (2002).
Scientific calculations are implemented through Astropy, not a new ML model.
"""
from pathlib import Path
import argparse, json
import numpy as np
import matplotlib.pyplot as plt

def simulate_light_curve(seed=12600, planet=True, period_days=3.25,
                         depth=0.004, duration_days=0.16, noise=0.001):
    rng = np.random.default_rng(seed)
    t = np.arange(0, 65, 1/48)
    flux = 1 + rng.normal(0,noise,len(t))
    if planet:
        phase = (t - 0.92 + period_days/2)%period_days-period_days/2
        flux[np.abs(phase)<duration_days/2] -= depth
    return t, flux

def detect(t,flux,min_period=2.8,max_period=3.8):
    """Return strongest *candidate*, which can occur even without a planet."""
    from astropy.timeseries import BoxLeastSquares
    periods = np.linspace(min_period,max_period,2000)
    durations = np.array([0.08,0.12,0.16,0.20])
    model = BoxLeastSquares(t,flux,dy=np.full(len(t),0.001))
    power = model.power(periods,durations,objective="snr")
    i = int(np.argmax(power.power))
    result = {"best_period_days":float(power.period[i]),
              "depth_ppm":float(power.depth[i])*1e6,
              "transit_duration_hours":float(power.duration[i])*24,
              "peak_snr_statistic":float(power.power[i])}
    return result,periods,np.asarray(power.power)

def run(planet=True,seed=12600,out="results/transits",depth=0.004):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    t,flux=simulate_light_curve(seed=seed,planet=planet,depth=depth)
    result,periods,scores=detect(t,flux)
    result.update({"source":"SIMULATED, not real Kepler observations",
                   "injected_period_days":3.25 if planet else None,
                   "period_error_days":abs(result["best_period_days"]-3.25) if planet else None})
    label="planet_injected" if planet else "noise_only"
    fig,axes=plt.subplots(2,1,figsize=(8,6))
    axes[0].scatter(t,flux,s=1.5)
    axes[0].set(xlabel="Time (days)",ylabel="Relative stellar flux",
                title="Simulated stellar brightness")
    axes[1].plot(periods,scores,lw=1)
    axes[1].axvline(result["best_period_days"],color="gray",linestyle="--")
    axes[1].set(xlabel="Trial period (days)",ylabel="BLS detection score",
                title="Box Least Squares periodicity search")
    fig.tight_layout();fig.savefig(out/(label+".png"),dpi=170);plt.close(fig)
    (out/(label+".json")).write_text(json.dumps(result,indent=2))
    return result

def sensitivity_study(out="results/sensitivity",trials=12):
    """Keep noise level fixed and compare recovery at three transit depths."""
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    rows=[]
    for depth_ppm in (4000,1500,500):
        recovered=0
        errors=[]
        for i in range(trials):
            t,y=simulate_light_curve(seed=12600+i,depth=depth_ppm/1e6)
            result,_,_=detect(t,y)
            error=abs(result["best_period_days"]-3.25)
            errors.append(error)
            recovered+=error<0.05
        rows.append({"depth_ppm":depth_ppm,"recovered":recovered,
                     "trials":trials,"median_period_error_days":float(np.median(errors))})
    import pandas as pd
    df=pd.DataFrame(rows)
    df.to_csv(out/"sensitivity_summary.csv",index=False)
    fig,ax=plt.subplots(figsize=(6,3.5))
    ax.bar(df.depth_ppm.astype(str),100*df.recovered/df.trials)
    ax.set(xlabel="Simulated transit depth (ppm)",ylabel="Correct period recovered (%)",
           ylim=(0,105),title="Sensitivity to weaker transits")
    fig.tight_layout();fig.savefig(out/"sensitivity.png",dpi=170);plt.close(fig)
    return df

if __name__=="__main__":
    p=argparse.ArgumentParser()
    p.add_argument("--output",default="results/transits")
    a=p.parse_args()
    print("Injected planet:",run(out=a.output))
    print("Noise-only:",run(planet=False,out=a.output))
    print(sensitivity_study())


In [ ]:
import exoplanet_detector as detector
from IPython.display import display, Image
positive=detector.run(out='results/transits')
negative=detector.run(planet=False,out='results/transits')
print('Injected transit:',positive)
print('Noise-only control:',negative)
display(Image(filename='results/transits/planet_injected.png'))
display(Image(filename='results/transits/noise_only.png'))


## 2. Controlled experiment: sensitivity to weaker simulated transits

The noise is held fixed and the injected transit depth is changed. This is an educational injection-recovery experiment, not a false-alarm calibration.


In [ ]:
sensitivity=detector.sensitivity_study(out='results/sensitivity',trials=12)
display(sensitivity)
display(Image(filename='results/sensitivity/sensitivity.png'))


## 3. Machine-learning extension: reduce misleading candidate flags

NASA Kepler DR25 catalog. Four astronomical measurements, one random forest, and one SNR-only baseline. The same star cannot appear in both training and test. Thresholds are set on validation stars. **No results should be reported until the real data run succeeds.**


In [ ]:
%%writefile ml_screening.py
"""Reproducible screening comparison on the historical Kepler DR25 KOI catalog.

This script predicts NASA's past Robovetter labels, NOT independently confirmed planets.
Generated with AI assistance; review AI_USE_LOG.md before citing results.
"""
from pathlib import Path
from urllib.parse import urlencode
import argparse, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

FEATURES = ["koi_period", "koi_duration", "koi_depth", "koi_model_snr"]
COLS = ["kepid", "kepoi_name", "koi_pdisposition"] + FEATURES
TABLE = "q1_q17_dr25_koi"

def nasa_url():
    query = "select " + ",".join(COLS) + " from " + TABLE
    return "https://exoplanetarchive.ipac.caltech.edu/TAP/sync?" + urlencode({"query": query, "format": "csv"})

def load_data(path, download=False):
    path = Path(path)
    if not path.exists():
        if not download:
            raise FileNotFoundError("Download NASA CSV and save to " + str(path) + "\nURL: " + nasa_url())
        r = requests.get(nasa_url(), timeout=80)
        r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(r.content)
    df = pd.read_csv(path, comment="#")
    missing = set(COLS) - set(df.columns)
    if missing:
        raise ValueError("Missing columns: " + str(missing))
    df = df.loc[df.koi_pdisposition.isin(["CANDIDATE", "FALSE POSITIVE"]), COLS].copy()
    df = df.dropna(subset=["kepid", "kepoi_name"]).drop_duplicates("kepoi_name")
    df["label"] = (df.koi_pdisposition == "CANDIDATE").astype(int)
    for col in FEATURES:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df.loc[~np.isfinite(df[col]) | (df[col] < 0), col] = np.nan
    if len(df) < 500 or df.label.nunique() != 2:
        raise ValueError("Unexpected NASA catalog or insufficient labeled observations")
    return df.reset_index(drop=True)

def split_by_star(df, seed=12600):
    y, groups = df.label.to_numpy(), df.kepid.to_numpy()
    allidx = np.arange(len(df))
    split1 = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    trainval, test = next(split1.split(allidx, y, groups))
    split2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed + 1)
    a, b = next(split2.split(trainval, y[trainval], groups[trainval]))
    train, valid = trainval[a], trainval[b]
    for left, right in [(train, valid), (train, test), (valid, test)]:
        assert not set(groups[left]).intersection(groups[right]), "Same-star data leakage!"
    for ix in (train, valid, test):
        assert len(np.unique(y[ix])) == 2
    return train, valid, test

def cutoff(scores, labels, desired=0.90):
    positives = np.asarray(scores)[np.asarray(labels) == 1]
    if not len(positives):
        raise ValueError("Missing positive validation cases")
    k = int(np.ceil(desired * len(positives))) - 1
    return float(np.sort(positives)[::-1][k])

def metrics(y, pred):
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    return dict(accuracy=float(accuracy_score(y, pred)),
                recall=float(recall_score(y, pred, zero_division=0)),
                precision=float(precision_score(y, pred, zero_division=0)),
                false_positive_rate=float(fp / (fp + tn)),
                fraction_flagged=float(np.mean(pred)),
                true_positive=int(tp), false_positive=int(fp),
                true_negative=int(tn), false_negative=int(fn))

def experiment(df, output="results/ml"):
    out = Path(output); out.mkdir(parents=True, exist_ok=True)
    train, valid, test = split_by_star(df)
    x = np.log1p(df[FEATURES].clip(lower=0).astype(float))
    y = df.label.to_numpy()
    forest = make_pipeline(
        SimpleImputer(strategy="median"),
        RandomForestClassifier(n_estimators=120, max_depth=9,
                               min_samples_leaf=3, random_state=12600, n_jobs=-1))
    forest.fit(x.iloc[train], y[train])
    snr_median = float(np.nanmedian(x.iloc[train].koi_model_snr))
    def snr(ix):
        return x.iloc[ix].koi_model_snr.fillna(snr_median).to_numpy()
    alternatives = {
       "SNR only": (snr(valid), snr(test)),
       "Random forest": (forest.predict_proba(x.iloc[valid])[:, 1],
                         forest.predict_proba(x.iloc[test])[:, 1]),
    }
    report = {}; rows = []
    for name, (vscore, tscore) in alternatives.items():
        selected = cutoff(vscore, y[valid])
        vpred = vscore >= selected
        tpred = tscore >= selected
        report[name] = {"threshold": selected,
                        "validation": metrics(y[valid], vpred),
                        "test": metrics(y[test], tpred)}
        rows.append({"method": name, **report[name]["test"]})
        if name == "Random forest":
            cm = confusion_matrix(y[test], tpred, labels=[0,1])
            fig, ax = plt.subplots(figsize=(5,4))
            ax.imshow(cm, cmap="Blues")
            ax.set_xticks([0,1], ["Screen out","Review"])
            ax.set_yticks([0,1], ["NASA false positive","NASA candidate"])
            for (i,j), count in np.ndenumerate(cm):
                ax.text(j,i,str(count),ha="center",va="center")
            ax.set_title("Holdout-star confusion matrix")
            fig.tight_layout(); fig.savefig(out/"confusion_matrix.png",dpi=170); plt.close(fig)
    table = pd.DataFrame(rows)
    table.to_csv(out/"comparison.csv",index=False)
    fig, ax = plt.subplots(figsize=(7,3))
    ax.barh(table.method, 100*table.fraction_flagged)
    ax.set_xlim(0,100)
    ax.set_xlabel("Test signals flagged (%)")
    ax.set_title("SNR threshold versus random-forest screening")
    fig.tight_layout(); fig.savefig(out/"screening_comparison.png",dpi=170); plt.close(fig)
    payload = {"dataset": TABLE, "n_rows": len(df),
               "n_train": len(train), "n_validation": len(valid), "n_test": len(test),
               "label_caveat": "Historical Robovetter classification; not confirmed planets",
               "features": FEATURES, "results": report}
    (out/"metrics.json").write_text(json.dumps(payload,indent=2))
    return payload

if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--csv",default="data/kepler_dr25.csv")
    parser.add_argument("--download",action="store_true")
    parser.add_argument("--output",default="results/ml")
    a = parser.parse_args()
    try:
        result = experiment(load_data(a.csv,a.download),a.output)
        print(json.dumps(result,indent=2))
    except Exception as exc:
        print("NASA test not completed:",str(exc))
        print("Manual CSV link:",nasa_url())
        raise


In [ ]:
from pathlib import Path
import ml_screening
csv=Path('data/kepler_dr25.csv')
print('Official NASA CSV URL: ',ml_screening.nasa_url())
try:
    df=ml_screening.load_data(csv,download=True)
    results=ml_screening.experiment(df,'results/ml')
    print('REAL NASA RESULTS:',results)
    display(Image(filename='results/ml/screening_comparison.png'))
    display(Image(filename='results/ml/confusion_matrix.png'))
except Exception as exc:
    print('NASA ANALYSIS NOT COMPLETED:',str(exc))
    print('If automatic download fails, open the URL above in your browser and upload its CSV:')
    try:
        from google.colab import files
        uploaded=files.upload()
        if uploaded:
            csv.parent.mkdir(parents=True,exist_ok=True)
            csv.write_bytes(next(iter(uploaded.values())))
            df=ml_screening.load_data(csv)
            results=ml_screening.experiment(df,'results/ml')
            print('REAL NASA RESULTS:',results)
    except Exception as err:
        print('Manual CSV fallback did not complete:',str(err))


## 4. Download evidence and write your own interpretation

Clearly distinguish simulated results from NASA catalog results. Report held-out candidate recall, false-positive rate, and fraction flagged for review. Explain what AI wrote, and what you checked independently.


In [ ]:
import shutil
from IPython.display import FileLink
result_zip=shutil.make_archive('ASTR_exoplanet_results','zip','results')
display(FileLink(result_zip))
